Import Libraries

In [25]:
import pandas as pd
import numpy as np
import json
import re
import torch
import os, sys, time, math
from sklearn.model_selection import train_test_split
print("GPU available:", torch.cuda.is_available())

GPU available: False


Buiding of Multi-label dataset

In [26]:
DATA_ROOT = "Cyberbullying_Datasets"
PROCESSED = "Processed_splits"
#MODEL_OUT = "roberta_out" # trained model is saved here

assert os.path.isdir(DATA_ROOT), f"DATA_ROOT not found: {DATA_ROOT}"
print(os.listdir(DATA_ROOT))

['Andrewmvd_Dataset', 'HateXplain', 'Saurabh_Shahane_Dataset']


In [27]:
OUT = PROCESSED
ROOT = DATA_ROOT
os.makedirs(OUT, exist_ok=True)


SH = os.path.join(ROOT, "Saurabh_Shahane_Dataset")

LABELS = ["cyberbullying", "hate_speech", "offensive", "gender", "ethnicity", "religion", "age", "sexual_orientation"]
NL = len(LABELS)
IDX = {l: i for i, l in enumerate(LABELS)}


def norm_text(s):
    s = str(s)
    s = re.sub(r"https?://\S+|www\.\S+", "<url>", s)
    s = re.sub(r"@\w+", "<user>", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def key(s):  # used only for de-duplication
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9 ]", " ", s.lower())).strip()


def row(text, source, pos=(), known=()):
    """pos: labels that are 1. known: labels whose value is known. Rule: a non-abusive text is known-negative for every label."""
    y = np.zeros(NL, dtype=int)
    m = np.zeros(NL, dtype=int)
    if "cyberbullying" not in pos:
        m[:] = 1
    else:
        for l in set(pos) | set(known) | {"cyberbullying"}:
            m[IDX[l]] = 1
        for l in pos:
            y[IDX[l]] = 1
    return {"text": norm_text(text), "source": source,
            **{l: y[i] for i, l in enumerate(LABELS)},
            **{f"m_{l}": m[i] for i, l in enumerate(LABELS)}}


rows, log = [], []

1. Andrewmvd Cyberbullying Classification Dataset

In [28]:
an = pd.read_csv(os.path.join(ROOT, "Andrewmvd_Dataset", "cyberbullying_tweets.csv"))
n0 = len(an)
an["k"] = an.tweet_text.map(lambda s: key(norm_text(s)))
conflict = an.groupby("k").cyberbullying_type.transform("nunique") > 1
an = an[~conflict]
an = an[an.cyberbullying_type != "other_cyberbullying"]
log.append(f"Andrewmvd: {n0} rows -> dropped {conflict.sum()} conflicting duplicates and "
           f"the 'other_cyberbullying' class -> {len(an)} kept")
A_TARGETS = ["gender", "ethnicity", "religion", "age"]

# Andrewmvd gives ONE type per tweet. If True, the other target labels of an abusive tweet are treated as
# UNKNOWN (mask 0) instead of "absent", so the model is not taught that a message can only have one type.
ANDREW_OTHERS_UNKNOWN = True

for t, y in zip(an.tweet_text, an.cyberbullying_type):
    if y == "not_cyberbullying":
        rows.append(row(t, "andrewmvd"))
    else:
        rows.append(row(t, "andrewmvd", pos=["cyberbullying", y], known=[] if ANDREW_OTHERS_UNKNOWN else A_TARGETS))


2. HateXplain (aggregated file)

In [29]:
hx = pd.read_csv(os.path.join(ROOT, "HateXplain", "final_hateXplain.csv"))
NONE = {"No_race", "Nonreligious", "No_gender", "No_orientation"}
for _, r in hx.iterrows():
    if r.label == "normal":
        rows.append(row(r.comment, "hatexplain"))
        continue
    pos = ["cyberbullying", "hate_speech" if r.label == "hatespeech" else "offensive"]
    if r.Gender not in NONE: pos.append("gender")
    if r.Race not in NONE: pos.append("ethnicity")
    if r.Religion not in NONE: pos.append("religion")
    if r["Sexual Orientation"] not in NONE: pos.append("sexual_orientation")
    rows.append(row(r.comment, "hatexplain", pos=pos, known=["hate_speech", "offensive", "gender", "ethnicity", "religion", "sexual_orientation"]))
log.append(f"HateXplain: {len(hx)} rows kept (aggregated file; per-annotator file not used)")

3. Saurabh Shahane Cyberbullying Dataset

In [30]:
# Shahane Twitter (racism + sexism union; most rows also in Andrewmvd)
tw = pd.read_csv(os.path.join(SH, "twitter_parsed_dataset.csv")).dropna(subset=["Annotation"])
for t, a in zip(tw.Text, tw.Annotation):
    if a == "none":
        rows.append(row(t, "shahane_twitter"))
    elif a == "sexism":
        rows.append(row(t, "shahane_twitter", pos=["cyberbullying", "gender"]))
    else:  # 'racism' targets are ambiguous (mostly religion per Andrewmvd relabelling)
        rows.append(row(t, "shahane_twitter", pos=["cyberbullying"]))
log.append(f"Shahane twitter_parsed: {len(tw)} rows before cross-source de-duplication "
           "(twitter_racism/twitter_sexism files are subsets, not used)")


# Shahane Kaggle insults
kg = pd.read_csv(os.path.join(SH, "kaggle_parsed_dataset.csv"))
def clean_kaggle(s):  # texts contain literal escapes such as \xa0, \\n, ạ
    s = str(s).strip().strip('"')
    s = re.sub(r"\\+(x[0-9a-fA-F]{2}|u[0-9a-fA-F]{4}|[ntr])", " ", s)
    return s.replace("\\'", "'")

for t, y in zip(kg.Text.map(clean_kaggle), kg.oh_label):
    rows.append(row(t, "shahane_kaggle", pos=["cyberbullying", "offensive"]) if y == 1
                else row(t, "shahane_kaggle"))
log.append(f"Shahane kaggle (insults): {len(kg)} rows")


# Shahane YouTube
yt = pd.read_csv(os.path.join(SH, "youtube_parsed_dataset.csv"))
for t, y in zip(yt.Text, yt.oh_label):
    rows.append(row(t, "shahane_youtube", pos=["cyberbullying"]) if y == 1
                else row(t, "shahane_youtube"))
log.append(f"Shahane youtube: {len(yt)} rows (one comment per user, so no history)")
log.append("Shahane aggression/attack/toxicity (Wikipedia talk pages): NOT used")


In [31]:
# Merge + de-duplicate across sources
df = pd.DataFrame(rows)
df = df[df.text.str.len() >= 5]
df["k"] = df.text.map(key)
before = len(df)
# keep the first occurrence; source order above gives priority Andrewmvd > HateXplain > others
df = df.drop_duplicates("k").drop(columns="k").reset_index(drop=True)
log.append(f"Cross-source + within-source duplicates removed: {before - len(df)}")
log.append(f"FINAL rows: {len(df)}")


# Split (stratified by source x cyberbullying)
strat = df.source + "_" + df.cyberbullying.astype(str)
train, temp = train_test_split(df, test_size=0.2, random_state=42, stratify=strat)
val, test = train_test_split(temp, test_size=0.5, random_state=42, stratify=strat.loc[temp.index])
for name, d in [("train", train), ("val", val), ("test", test)]:
    d.to_csv(os.path.join(OUT, f"{name}.csv"), index=False)


# Statistics
Y = df[LABELS].values; M = df[[f"m_{l}" for l in LABELS]].values
stats = pd.DataFrame({"positives": (Y * M).sum(0), "known_rows": M.sum(0), "unknown_rows": (1 - M).sum(0)}, index=LABELS)
pos_rows = df[df.cyberbullying == 1]
tcols = ["hate_speech", "offensive", "gender", "ethnicity", "religion", "age", "sexual_orientation"]
n_labels = pos_rows[tcols].sum(1) + 1
co = df[LABELS].T.dot(df[LABELS])

with open(os.path.join(OUT, "data_report.txt"), "w") as f:
    for line in log:
        f.write(line + "\n")
    f.write(f"\nSplit sizes: train={len(train)} val={len(val)} test={len(test)}\n")
    f.write("\nRows per source:\n" + df.groupby("source").cyberbullying.agg(["count", "mean"])
            .rename(columns={"mean": "share_abusive"}).round(3).to_string() + "\n")
    f.write("\nPer-label counts:\n" + stats.to_string() + "\n")
    f.write("\nLabels per abusive row (incl. 'cyberbullying'):\n" +
            n_labels.value_counts().sort_index().to_string() + "\n")
    f.write(f"Share of abusive rows with 3+ labels: {(n_labels >= 3).mean():.3f}\n")
    f.write("\nCo-occurrence matrix:\n" + co.to_string() + "\n")
print(open(os.path.join(OUT, "data_report.txt")).read())

Andrewmvd: 47692 rows -> dropped 3637 conflicting duplicates and the 'other_cyberbullying' class -> 38043 kept
HateXplain: 20109 rows kept (aggregated file; per-annotator file not used)
Shahane twitter_parsed: 16848 rows before cross-source de-duplication (twitter_racism/twitter_sexism files are subsets, not used)
Shahane kaggle (insults): 8799 rows
Shahane youtube: 3464 rows (one comment per user, so no history)
Shahane aggression/attack/toxicity (Wikipedia talk pages): NOT used
Cross-source + within-source duplicates removed: 7945
FINAL rows: 79297

Split sizes: train=63437 val=7930 test=7930

Rows per source:
                 count  share_abusive
source                               
andrewmvd        37605          0.833
hatexplain       20094          0.611
shahane_kaggle    8652          0.315
shahane_twitter   9493          0.145
shahane_youtube   3453          0.120

Per-label counts:
                    positives  known_rows  unknown_rows
cyberbullying           48137       792

C:\Users\yenul\AppData\Local\Temp\ipykernel_15160\3502362232.py:25: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  n_labels = pos_rows[tcols].sum(1) + 1


Baselines (TF-IDF features + Logistic Regression and Naive Bayes)

In [33]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB

MCOLS = [f"m_{l}" for l in LABELS]
tr = pd.read_csv(f"{PROCESSED}/train.csv")
te = pd.read_csv(f"{PROCESSED}/test.csv")
print(tr.shape, te.shape)

def masked_scores(y, p, m):
    """Score only where the label is known (mask = 1)."""
    per, prec, rec = {}, {}, {}
    tp_all = fp_all = fn_all = 0
    for j, l in enumerate(LABELS):
        k = m[:, j] == 1
        yt, yp = y[k, j], p[k, j]
        tp = int((yt & yp).sum()); fp = int((~yt & yp).sum()); fn = int((yt & ~yp).sum())
        tp_all += tp; fp_all += fp; fn_all += fn
        per[l]  = round(2 * tp / max(2 * tp + fp + fn, 1), 4)
        prec[l] = round(tp / max(tp + fp, 1), 4)
        rec[l]  = round(tp / max(tp + fn, 1), 4)
    return {"micro_f1": round(2 * tp_all / (2 * tp_all + fp_all + fn_all), 4),
            "macro_f1": round(float(np.mean(list(per.values()))), 4),
            "hamming_loss": round(float((y != p)[m == 1].mean()), 4),
            "per_label_f1": per, "per_label_precision": prec, "per_label_recall": rec}

(63437, 18) (7930, 18)


In [34]:
# TF-IDF turns each message into numbers based on which words/word pairs it contains
vec = TfidfVectorizer(max_features=50000, ngram_range=(1, 2), min_df=2, sublinear_tf=True)
Xtr = vec.fit_transform(tr.text.astype(str))
Xte = vec.transform(te.text.astype(str))

def run(make_model):
    P = np.zeros((len(te), len(LABELS)), dtype=bool)
    for j, l in enumerate(LABELS):
        k = tr[f"m_{l}"].values == 1
        model = make_model()
        model.fit(Xtr[k], tr[l].values[k])
        P[:, j] = model.predict(Xte) == 1
    return masked_scores(te[LABELS].values.astype(bool), P, te[MCOLS].values)

res_lr = run(lambda: LogisticRegression(max_iter=2000, class_weight="balanced"))
res_nb = run(lambda: MultinomialNB(alpha=0.1))
for name, r in [("Logistic Regression", res_lr), ("Naive Bayes", res_nb)]:
    print(name, {k: r[k] for k in ["micro_f1", "macro_f1", "hamming_loss"]})

Logistic Regression {'micro_f1': 0.8124, 'macro_f1': 0.7776, 'hamming_loss': 0.1043}
Naive Bayes {'micro_f1': 0.7694, 'macro_f1': 0.6257, 'hamming_loss': 0.118}


In [35]:
f1 = pd.DataFrame({"Naive Bayes": res_nb["per_label_f1"], "Logistic Regression": res_lr["per_label_f1"]})
display(f1)
print("Macro-F1 over the 7 specific labels:\n", f1.drop(index="cyberbullying").mean().round(4))
json.dump({"logistic_regression": res_lr, "naive_bayes": res_nb}, open(f"{PROCESSED}/baseline_results.json", "w"), indent=2)

,Naive Bayes,Logistic Regression
cyberbullying,0.8455,0.8660
hate_speech,0.5424,0.7050
offensive,0.3142,0.5425
gender,0.6824,0.7487
ethnicity,0.7720,0.8553
religion,0.7771,0.8586
age,0.9275,0.9588
sexual_orientation,0.1442,0.6859


Macro-F1 over the 7 specific labels:
 Naive Bayes            0.5943
Logistic Regression    0.7650
dtype: float64
